# Multi-Head Attention mit Transformer-Encoder-Logik

## Gemini Code

In [11]:
import tensorflow as tf
from tensorflow.keras import layers

class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        # 1. Multi-Head Attention Layer
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

        # 2. Feed Forward Network (FFN)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])

        # 3. Layer Normalization
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

        # 4. Dropout for regularization
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False, mask=None):
        # Multi-Head Attention layer requires Query, Value, and Key.
        # For self-attention, all three are the same (inputs).
        attn_output = self.att(inputs, inputs, inputs, attention_mask=mask)
        attn_output = self.dropout1(attn_output, training=training)

        # Residual connection 1
        out1 = self.layernorm1(inputs + attn_output)

        # Feed Forward network
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)

        # Residual connection 2
        return self.layernorm2(out1 + ffn_output)

In [12]:
# Hyperparameters
vocab_size = 5000  # Size of vocabulary
max_len = 100      # Max length of input sequence
embed_dim = 64     # Embedding size for each token
num_heads = 4      # Number of attention heads
ff_dim = 128       # Hidden layer size in feed-forward network

# Model Definition
inputs = layers.Input(shape=(max_len,))
embedding_layer = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
x = embedding_layer(inputs)

# Add the Transformer Block
transformer_block = TransformerEncoderBlock(embed_dim, num_heads, ff_dim)
x = transformer_block(x)

# Classification Head (Example)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(2, activation="softmax")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 100, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_block       │ (None, 100, 64)        │        83,200 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 403,330 (1.54 MB)

 Trainable params: 403,330 (1.54 MB)

 Non-trainable params: 0 (0.00 B)

## Mein Code

In [10]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import MultiHeadAttention, Input, Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.layers import Dropout, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.layers import TextVectorization

# 1. Hyperparameter definieren
VOCAB_SIZE = 1000       # Maximale Anzahl an Wörtern im Wörterbuch
MAX_LEN = 10            # Maximale Länge eines Satzes
EMBEDDING_DIM = 300     # Dimension deiner Wortvektoren
FEED_FORWARD_DIM = 64
DROPOUT_RATE = 0.1

# =====================================================================
# 2. Beispiel-Textdaten erstellen (Klassifikation: Positiv/Negativ)
# =====================================================================
texts = [
    "Das Produkt ist absolut fantastisch und super",
    "Ich liebe diesen Service, wirklich toll gemacht",
    "Der größte Müll, nie wieder kaufe ich das",
    "Schlechte Qualität und der Support ist schrecklich",
    "Total genial, hat mir sehr geholfen",
    "Das war reine Geldverschwendung und dumm"
]
# Labels: 1 = Positiv, 0 = Negativ
labels = np.array([1, 1, 0, 0, 1, 0], dtype=np.int32)

# Text-Vektorisierung (Wörter in Zahlen umwandeln)
vectorize_layer = TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=MAX_LEN)
vectorize_layer.adapt(texts)

# Texte in Arrays umwandeln
X_train = vectorize_layer(np.array(texts))
y_train = labels

# =====================================================================
# 3. Das Transformer-Modell (Functional API)
# =====================================================================
inputs = Input(shape=(MAX_LEN,))

# Wort-Embedding (Zahlen -> Vektoren)
x = Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM)(inputs)

# --- DEIN TRANSFORMER BLOCK (Korrigiert für Residual Connections) ---
# MultiHeadAttention benötigt: query, value, key (hier 3x 'x' für Self-Attention)
attention_output = MultiHeadAttention(num_heads=8, key_dim=EMBEDDING_DIM)(x, x, x)
attention_output = Dropout(DROPOUT_RATE)(attention_output)
x1 = LayerNormalization(epsilon=1e-6)(x + attention_output) # Residual-Add + Norm

# Feed-Forward Network
ffn_output = Dense(FEED_FORWARD_DIM, activation="relu")(x1)
ffn_output = Dense(EMBEDDING_DIM)(ffn_output)
ffn_output = Dropout(DROPOUT_RATE)(ffn_output)
x2 = LayerNormalization(epsilon=1e-6)(x1 + ffn_output) # Residual-Add + Norm
# --------------------------------------------------------------------

# Klassifikations-Kopf (Umwandlung in eine finale Vorhersage)
x3 = GlobalAveragePooling1D()(x2) # Reduziert die Sequenzlänge auf einen Vektor
outputs = Dense(1, activation="sigmoid")(x3) # 1 Output für Binäre Klassifikation (0 oder 1)

# Modell instanziieren
model = Model(inputs=inputs, outputs=outputs)

# =====================================================================
# 4. Kompilieren und Trainieren
# =====================================================================
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

print("\nStarte Training...")
model.fit(X_train, y_train, epochs=10, batch_size=2)

# =====================================================================
# 5. Testen mit einem neuen Satz
# =====================================================================
test_text = ["Der Service war super"]
test_vector = vectorize_layer(np.array(test_text))
prediction = model.predict(test_vector)

print(f"\nSatz: '{test_text[0]}'")
print(f"Wahrscheinlichkeit für 'Positiv': {prediction[0][0]:.4f}")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 10, 300)   │    300,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 300)   │  2,887,500 │ embedding[0][0],  │
│ (MultiHeadAttentio… │                   │            │ embedding[0][0],  │
│                     │                   │            │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 10, 300)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 300)   │          0 │ embedding[0][0],  │
│                     │                   │            │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 300)   │        600 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 10, 64)    │     19,264 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 10, 300)   │     19,500 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 10, 300)   │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 10, 300)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 300)   │        600 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 300)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 1)         │        301 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,227,765 (12.31 MB)

 Trainable params: 3,227,765 (12.31 MB)

 Non-trainable params: 0 (0.00 B)


Starte Training...
Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5000 - loss: 4.0392 
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6667 - loss: 0.7955
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5000 - loss: 1.5289
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8333 - loss: 0.3026
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8333 - loss: 0.3026
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0229
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.0246
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0251    
Epoch 9/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.0154    
Epoch 10/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step

Satz: 'Der Service war super'
Wahrscheinlichkeit für 'Positiv': 0.9112
